#1. Leitura da Bronze persistida
Começamos lendo as tabelas persistidas, e não novamente os CSVs. Isso é importante para mostrar que existe realmente um pipeline entre camadas.

In [0]:
pokemon_bronze = spark.table(
    "mvp_data_engineering.bronze.pokemon_raw"
)

combats_bronze = spark.table(
    "mvp_data_engineering.bronze.combats_raw"
)

tests_bronze = spark.table(
    "mvp_data_engineering.bronze.tests_raw"
)

print("Pokémon:", pokemon_bronze.count())
print("Combates:", combats_bronze.count())
print("Tests:", tests_bronze.count())

Pokémon: 800
Combates: 50000
Tests: 10000


#2. Contrução da silver.pokemon

##2.1 Padronização da tabela

Etapa para garantir o tipo de dado de cada coluna e que o nome esteja em snake_case.

In [0]:

#Padronizando os nomes das colunas e tipagem explícita
from pyspark.sql.functions import col

pokemon_silver = pokemon_bronze.select(
    col("`#`").cast("int").alias("pokemon_id"),
    col("`Name`").cast("string").alias("name"),
    col("`Type 1`").cast("string").alias("type_1"),
    col("`Type 2`").cast("string").alias("type_2"),
    col("`HP`").cast("int").alias("hp"),
    col("`Attack`").cast("int").alias("attack"),
    col("`Defense`").cast("int").alias("defense"),
    col("`Sp. Atk`").cast("int").alias("sp_attack"),
    col("`Sp. Def`").cast("int").alias("sp_defense"),
    col("`Speed`").cast("int").alias("speed"),
    col("`Generation`").cast("int").alias("generation"),
    col("`Legendary`").cast("boolean").alias("is_legendary"),
    col("_ingestion_timestamp"),
    col("_source_file")
)

##2.2 Verificando linhas inválidas

In [0]:
#verificação de invalidade dos pokémon listados
invalid_pokemon = pokemon_silver.filter(
    col("pokemon_id").isNull()
    | col("type_1").isNull()
    | col("hp").isNull()
    | col("attack").isNull()
    | col("defense").isNull()
    | col("sp_attack").isNull()
    | col("sp_defense").isNull()
    | col("speed").isNull()
    | col("generation").isNull()
    | col("is_legendary").isNull()
    | (col("hp") < 0)
    | (col("attack") < 0)
    | (col("defense") < 0)
    | (col("sp_attack") < 0)
    | (col("sp_defense") < 0)
    | (col("speed") < 0)
)

In [0]:
print(
    "Pokémon considerados inválidos:",
    invalid_pokemon.count()
)

Pokémon considerados inválidos: 0


##2.3 Verificando duplicatas

In [0]:
#Removendo duplicatas
before = pokemon_silver.count()

pokemon_silver = pokemon_silver.dropDuplicates(
    ["pokemon_id"]
)

after = pokemon_silver.count()

print("Antes:", before)
print("Depois:", after)
print("Duplicados removidos:", before - after)

Antes: 800
Depois: 800
Duplicados removidos: 0


A análise de unicidade não identificou identificadores duplicados, portanto nenhuma observação foi removida por essa regra.

#3. Contrução da silver.combats

##3.1 Padronização da tabela
Etapa para garantir o tipo de dado de cada coluna e que o nome esteja em snake_case.

In [0]:
combats_silver = combats_bronze.select(
    col("First_pokemon").cast("int").alias("first_pokemon_id"),
    col("Second_pokemon").cast("int").alias("second_pokemon_id"),
    col("Winner").cast("int").alias("winner_id"),
    col("_ingestion_timestamp"),
    col("_source_file")
)

##3.2 Criação das colunas "battle_id", "winner_id", "loser_id" e "first_pokemon_won"

Enriquecendo a tabela para termos chaves referências de cada batalha, facilitando análises futuras

In [0]:
#Criando coluna battle_id
from pyspark.sql.functions import monotonically_increasing_id

combats_silver = combats_silver.withColumn(
    "battle_id",
    monotonically_increasing_id()
)

In [0]:
#Criando vencedor/perdedor
from pyspark.sql.functions import when

combats_silver = (
    combats_silver
    .withColumn(
        "loser_id",
        when(
            col("winner_id") == col("first_pokemon_id"),
            col("second_pokemon_id")
        ).otherwise(col("first_pokemon_id"))
    )
    .withColumn(
        "first_pokemon_won",
        col("winner_id") == col("first_pokemon_id")
    )
)

display(
    combats_silver.select(
        "battle_id",
        "first_pokemon_id",
        "second_pokemon_id",
        "winner_id",
        "loser_id",
        "first_pokemon_won"
    ).limit(10)
)

battle_id,first_pokemon_id,second_pokemon_id,winner_id,loser_id,first_pokemon_won
0,266,298,298,266,false
1,702,701,701,702,false
2,191,668,668,191,false
3,237,683,683,237,false
4,151,231,151,231,true
5,657,752,657,752,true
6,192,134,134,192,false
7,73,545,545,73,false
8,220,763,763,220,false
9,302,31,31,302,false


##3.3 Verificação de consistência das colunas

In [0]:
#Verificando se o vencedor está na batalha
invalid_winners = combats_silver.filter(
    (col("winner_id") != col("first_pokemon_id"))
    & (col("winner_id") != col("second_pokemon_id"))
)

print(
    "Vencedores inconsistentes:",
    invalid_winners.count()
)

Vencedores inconsistentes: 0


In [0]:
#Verificando se há batalhas entre o mesmo Pokémon
self_battles = combats_silver.filter(
    col("first_pokemon_id")
    == col("second_pokemon_id")
)

print(
    "Self battles:",
    self_battles.count()
)

Self battles: 0


In [0]:
#Verificando se há referências inválidas nos IDs
pokemon_ids = pokemon_silver.select("pokemon_id")

invalid_first = (
    combats_silver
    .join(
        pokemon_ids,
        combats_silver.first_pokemon_id
        == pokemon_ids.pokemon_id,
        "left_anti"
    )
)

invalid_second = (
    combats_silver
    .join(
        pokemon_ids,
        combats_silver.second_pokemon_id
        == pokemon_ids.pokemon_id,
        "left_anti"
    )
)

invalid_winner_fk = (
    combats_silver
    .join(
        pokemon_ids,
        combats_silver.winner_id
        == pokemon_ids.pokemon_id,
        "left_anti"
    )
)

print("First inválidos:", invalid_first.count())
print("Second inválidos:", invalid_second.count())
print("Winner inválidos:", invalid_winner_fk.count())

First inválidos: 0
Second inválidos: 0
Winner inválidos: 0


#4. Contrução da silver.tests

In [0]:
tests_silver = tests_bronze.select(
    col("First_pokemon").cast("int").alias("first_pokemon_id"),
    col("Second_pokemon").cast("int").alias("second_pokemon_id"),
    col("_ingestion_timestamp"),
    col("_source_file")
)

tests_silver = tests_silver.withColumn(
    "test_id",
    monotonically_increasing_id()
)

display(tests_silver.limit(10))

first_pokemon_id,second_pokemon_id,_ingestion_timestamp,_source_file,test_id
129,117,2026-09-27T00:43:27.335Z,tests.csv,0
660,211,2026-09-27T00:43:27.335Z,tests.csv,1
706,115,2026-09-27T00:43:27.335Z,tests.csv,2
195,618,2026-09-27T00:43:27.335Z,tests.csv,3
27,656,2026-09-27T00:43:27.335Z,tests.csv,4
126,222,2026-09-27T00:43:27.335Z,tests.csv,5
436,207,2026-09-27T00:43:27.335Z,tests.csv,6
121,36,2026-09-27T00:43:27.335Z,tests.csv,7
169,636,2026-09-27T00:43:27.335Z,tests.csv,8
302,688,2026-09-27T00:43:27.335Z,tests.csv,9


#5. Persistindo a camada Silver

In [0]:
(
    pokemon_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.silver.pokemon")
)

(
    combats_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.silver.combats")
)

(
    tests_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.silver.tests")
)

In [0]:
%sql

SHOW TABLES IN mvp_data_engineering.silver;

database,tableName,isTemporary
silver,combats,false
silver,pokemon,false
silver,tests,false


#6. Validando Bronze X Silver

In [0]:
comparison = [
    (
        "pokemon",
        pokemon_bronze.count(),
        pokemon_silver.count()
    ),
    (
        "combats",
        combats_bronze.count(),
        combats_silver.count()
    ),
    (
        "tests",
        tests_bronze.count(),
        tests_silver.count()
    )
]

comparison_df = spark.createDataFrame(
    comparison,
    ["dataset", "bronze_records", "silver_records"]
)

display(comparison_df)

dataset,bronze_records,silver_records
pokemon,800,800
combats,50000,50000
tests,10000,10000


## Resumo das transformações Bronze → Silver

### Pokémon
- Padronização dos nomes das colunas para snake_case.
- Definição explícita dos tipos de dados.
- Validação de campos obrigatórios.
- Validação dos atributos numéricos.
- Verificação de unicidade de pokemon_id.
- Preservação de valores nulos em type_2, pois representam Pokémon que possuem apenas um tipo.

### Combates
- Padronização dos nomes das colunas.
- Definição explícita dos tipos.
- Criação do identificador battle_id.
- Criação de loser_id.
- Criação do indicador first_pokemon_won.
- Validação da integridade referencial com a entidade Pokémon.
- Validação de que winner_id corresponde a um dos participantes da batalha.
- Verificação de batalhas de um Pokémon contra ele mesmo.

### Test
- Padronização dos nomes das colunas.
- Definição explícita dos tipos.
- Criação de test_id.
- Validação dos identificadores dos Pokémon.